# Creating a chatbot for structured data using SQL

## Overview
**Generative AI (GenAI)** is a groundbreaking technology that generates human-like texts, images, code, and other forms of content. Although this is all true the focus of many GenAI techniques or implementations have been on unstructured data such as PDF's, text docs, image files, websites, etc. where it is required to set a parameter called *top K*. Top K utilizes an algorithm to only retrieve the top scored pieces of content or docs that is relevant to the users ask. This limits the amount of data the model is presented putting a disadvantage for users that may want to gather information from structured data like CSV and JSON files because they typically want all the occurrences relevant data appears. 

An example would be if you had a table that lists different types of apples, where they originate, and their colors and you want a list of red apples that originate from the US the model would only give you partial amount of the data you need because it is limited to looking for the top relevant data which may be limited to only finding the top 4 or 20 names of apples (depending on how you have configured your model) instead of listing them all. 

The technique that is laid our in this tutorial utilizes **SQL databases** and asks the model to create a query based on the ask of the user. It will then submit that query to the database and present the user with the results. This will not only give us all the information we need but will also decrease the chances of hitting our token limit.

## Prerequisites

We assume you have access to Azure AI Studio, Azure SQL Databases, and have already deployed an LLM. For this tutorial we used **gpt-5.4-nano** and used the **Python 3.10** kernel within our Azure Jupyter notebook.

## Learning objectives

In this tutorial you will learn:
- Setting up a Azure SQL database
- Creating a SQl table and query from it
- Creating a chatbot and utilizing langchains SQL agent to connect the bot to a database

## Get started

### Install packages <a id="packages"></a>

In [ ]:
!pip install -U openai langchain langchain-openai langchain-community tiktoken
!pip install -U pyodbc sqlalchemy pandas

## Setting up Python and Azure OpenAI environment

In [ ]:
# azure cloud login
!az login --identity

In [ ]:
# set your subscription
!az account set --subscription "NIH.CIT.CS.CloudLab.Azure_XXX" # your subscription name

In [ ]:
# confirm subscription
!az account show

In [ ]:
%pip install --quiet openai

### Get Azure OpenAI environment from Azure
In search bar window, type **Foundry** and select **Microsoft Foundry**

In microsoft foundry window, click on **resource** where you deployed gpt and embedding models

For Azure OpenAI endpoint, click rectangle under **Azure OpenAI endpoint**. It would copy OpenAI endpoint, replace it in below cell with **your_openai_endpoint**

For Azure OpenAI Key, click rectangle under **API key**. It would copy OpenAI key, replace it in below cell with **yourOpenAIKey**

### Azure infrastructure setup

In [ ]:
# configure environment
import os, time, json, numpy as np, pandas as pd
from typing import List, Dict
from xml.etree import ElementTree as ET
from openai import OpenAI
import requests

AZURE_OPENAI_ENDPOINT="your_openai_endpoint"
AZURE_OPENAI_KEY = "your_openAI_key"
AZURE_OPENAI_DEPLOYMENT_EMBED =  "text-embedding-3-small"
AZURE_OPENAI_DEPLOYMENT_CHAT =  "gpt-5.4-nano"

client = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_KEY)

In [ ]:

resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT_CHAT,
        messages=[{"role":"user","content":"Return the single word: ready"}],
        temperature=0,
    )
print("Azure OpenAI OK:", resp.choices[0].message.content)


### Create Azure SQL Database <a id="azure_db"></a>

Follow the instructions [here](https://learn.microsoft.com/en-us/azure/azure-sql/database/single-database-create-quickstart?view=azuresql&tabs=azure-portal) to create a single database in Azure SQL Database. 

**Note:** For this tutorials database the field name **'Use existing data'** was set to **None** and the **location** to **US East 2**.

 #### connect to Azure SQL database

In [ ]:
import os
import re
import sys
import platform
import pandas as pd
import pyodbc

# ---------- DIRECT ODBC HELPERS ----------
DRIVER = "{ODBC Driver 18 for SQL Server}"
SQL_SERVER="your_SQL_server"
SQL_DATABASE="your_SQL_database"
SQL_USER="your_username"
SQL_PASSWORD="your_password"

conn_str = (
    f"DRIVER={DRIVER};SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
    f"UID={SQL_USER};PWD={SQL_PASSWORD};Encrypt=yes;TrustServerCertificate=no;Connection Timeout=60;"
)


#### Load csv file and test SQL database connection

In [ ]:
# CSV + table constraints
CSV_FILENAME   = os.getenv("CSV_FILENAME", "/path_to_file/ds_salaries.csv")
RESTRICTED_TBL = "ds_salaries"
MAX_ROWS       = int(os.getenv("MAX_ROWS", "50"))

def get_conn():
    return pyodbc.connect(conn_str)

# Quick test query
cur = get_conn().cursor()
cur.execute(f"SELECT TOP (3) * FROM [{RESTRICTED_TBL}];")
rows = cur.fetchall()
print(f"SQL connectivity OK. Sample rows from '{RESTRICTED_TBL}':", len(rows))

This function retrieves the schema of a SQL Server table and prints a concise, nicely formatted summary showing each column along with its data type.

In [ ]:

def get_schema_text():
    with get_conn() as conn:
        cur = conn.cursor()
        cur.execute(f"""
            SELECT c.name AS column_name, TYPE_NAME(c.user_type_id) AS type_name
            FROM sys.columns c
            WHERE c.object_id = OBJECT_ID('{RESTRICTED_TBL}')
            ORDER BY c.column_id;
        """)
        rows = cur.fetchall()
        cols_text = ", ".join([f"{r.column_name} ({r.type_name})" for r in rows])
        return f"TABLE {RESTRICTED_TBL}: {cols_text}"

SCHEMA_TEXT = get_schema_text()
print("Schema summary:\n", SCHEMA_TEXT)


This function is designed to validate and safely rewrite SQL queries so they meet strict read‑only and table‑restriction rules. It prevents unsafe or unauthorized SQL operations and ensures the query only accesses the allowed table.

In [ ]:
import re

WRITE = re.compile(r"(INSERT|UPDATE|DELETE|DROP|ALTER|TRUNCATE|MERGE|CREATE|GRANT|REVOKE|EXEC|EXECUTE)", re.IGNORECASE)
TABLE = "ds_salaries"
MAX = 1000

def ensure_readonly_and_restricted(sql):
    sql = sql.strip()

    # Only one SELECT, no write commands
    if ";" in sql.rstrip(";"):
        raise ValueError("Only one SELECT allowed.")
    if WRITE.search(sql):
        raise ValueError("Only SELECT queries allowed.")
    if not sql.lower().startswith("select"):
        raise ValueError("Query must start with SELECT.")

    # Check table name in FROM or JOIN
    tables = re.findall(r"FROM\s+(\S+)|JOIN\s+(\S+)", sql, re.IGNORECASE)
    found = {t.split(".")[-1].strip("[]") for pair in tables for t in pair if t}
    if found != {TABLE}:
        raise ValueError("Query must use only the ds_salaries table.")

    # Add TOP if missing
    if "top" not in sql.lower():
        sql = re.sub(r"select", f"SELECT TOP {MAX}", sql, flags=re.IGNORECASE)

    return sql


This builds an AI‑driven SQL assistant that generates safe, single‑table, read‑only SQL queries, constrained by a strict system prompt and informed by your table’s schema.

In [ ]:

# ---------- PROMPT & SQL GENERATION ----------
SYSTEM_INSTRUCTIONS = f"""
You are a SQL assistant for a single table named '{RESTRICTED_TBL}' loaded from ds_salaries.csv.
Rules:
- Output ONLY one safe SQL SELECT statement targeting ONLY the '{RESTRICTED_TBL}' table.
- Never write, update, delete, or alter data.
- Use appropriate predicates and ordering.
- Include TOP {MAX_ROWS} for large result sets.
- Table schema:
{SCHEMA_TEXT}
"""

def generate_sql(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user",   "content": f"Question: {question}\nReturn ONLY the SQL SELECT."}
    ]
    resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT_CHAT,
        messages=messages,
        temperature=0,
    )
    sql = resp.choices[0].message.content.strip()
    return sql


This function takes a SQL query, validates it, executes it safely, converts the results into a DataFrame, and returns a bounded result set—ensuring both safety and convenience in downstream data analysis.

In [ ]:
# ---------- EXECUTE ----------
def execute_sql(sql: str) -> pd.DataFrame:
    safe = ensure_readonly_and_restricted(sql)
    with get_conn() as conn:
        cur = conn.cursor()
        cur.execute(safe)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    df = pd.DataFrame.from_records(rows, columns=cols)
    return df.head(MAX_ROWS)


The function is a wrapper that ties together your SQL‑generation, SQL‑validation, execution, and preview workflow. It allows a user to enter a natural‑language question and immediately see the generated SQL and a preview of the results.

In [ ]:
# ---------- CHAT ----------
def ask(question: str, show_sql: bool = True):
    print(f"\nQ: {question}")
    sql = generate_sql(question)
    if show_sql:
        print("\nGenerated SQL:\n", sql)
    try:
        df = execute_sql(sql)
        print("\nPreview:")
        print(df.to_string(index=False))
        return df
    except Exception as e:
        print("\nERROR:", e)
        return None


In [ ]:
# ---------- EXAMPLES ----------
# Adjust these examples to match the columns in your ds_salaries.csv
ask("Show the first 20 rows from ds_salaries.")
ask("What is the average salary by job_title, highest to lowest?")
ask("Count the number of records per company_size.")
ask("List the top 20 rows where remote_ratio = 100 ordered by salary in descending order.")

## Conclusion <a id="conclusion"></a>

In this notebook you learned how to set up a Azure SQL database and connect your model to the database, creating a chatbot that can read and retrieve data from structured data formats.

## Clean up <a id="cleanup"></a>

Dont forget to turn off or delete any notebooks or compute resources! Below you will find instructions to delete the SQL database. With the first step to close the connection to the database.

In [ ]:
get_conn().close()

 Next we will delete our database, wait for the command to output **'Finished'**.

In [ ]:
resource_group="your_resource_group"
!az sql db delete --name {SQL_DATABASE} --resource-group {resource_group} --server {SQL_SERVER} --yes

For this command you will need your subscriptions ID which can be found running the following command:

In [ ]:
!az sql server list --resource-group {resource_group}

Finally delete your SQL server, wait for the command to output **'Finished'**.

In [ ]:
subscription_id='your_subscription_id'
!az sql server delete --name {SQL_SERVER} --resource-group {resource_group} --subscription {subscription_id}  -y